# SEG Regularization Tuning — Tox21 Multi-Label Classification

Minimal SEG benchmark for multi-label toxicity classification (12 assays, NaN-masked BCE).

In [1]:
# === Setup ===
import sys
import random
from pathlib import Path
workspace_root = Path.cwd().parent
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

import numpy as np
import pandas as pd
import torch

def set_seed(seed: int) -> None:
    """Set all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"Workspace: {workspace_root}")

Workspace: c:\Users\robsc\Home\Dev\molfusion2


In [2]:
# === Configuration ===
# TUNE THESE for regularization experiments

CONFIG = {
    # === REPRODUCIBILITY ===
    "seed": 42,                   # Global seed for reproducibility (None to disable)
    
    # Architecture
    "hidden_channels": 128,
    "K": 4,
    "num_layers": 3,
    "pool": "set2set",
    "set2set_processing_steps": 6,
    
    # Fusion
    "fusion": "cross_mha",           # Options: "concat", "cross_mha", "gated", "film"
    "fusion_dim": 64,
    "fusion_n_heads": 8,
    "text_projection_dim": 64,      # Project 3072 → 64
    
    # === TEXT PROJECTION INITIALIZATION ===
    "text_proj_init": "xavier",      # "xavier" (scaled, recommended) or "kaiming" (PyTorch default)
    "text_proj_init_gain": 0.1,      # Gain for Xavier init (lower = smaller gradients)
    "freeze_text_proj": True,       # Freeze text projection (implicit regularization for small datasets)
    
    # === REGULARIZATION ===
    "dropout": 0.3,                  # Graph encoder dropout
    "fusion_dropout": 0.3,           # Fusion layer dropout
    "head_dropout": 0.6,             # Prediction head dropout (higher for multi-task)
    "weight_decay": 1e-1,            # L2 regularization
    
    # === HEAD ===
    "head_type": "mlp",              # Options: "mlp", "linear"
    "head_hidden_dim": 64,           # Larger for multi-task
    
    # Training
    "learning_rate": 3e-4,           # Lower LR for larger dataset
    "batch_size": 64,                # Larger batch for stability
    "num_epochs": 100,
    "patience": 20,
    
    # === LR SCHEDULER ===
    "scheduler": "cosine",
    "scheduler_patience": 5,
    "scheduler_factor": 0.5,
    "min_lr": 1e-6,
    
    # === GRADIENT CLIPPING ===
    "grad_clip": None,            # Max gradient norm (None to disable). Recommended: 1.0-5.0
    
    # Multi-label
    "num_tasks": 12,                 # Tox21 has 12 assays
}

# Set global seed for reproducibility
if CONFIG["seed"] is not None:
    set_seed(CONFIG["seed"])
    print(f"=== Seed: {CONFIG['seed']} (reproducibility enabled) ===")
else:
    print("=== Seed: None (non-deterministic) ===")

print("=== Tox21 Multi-Label Configuration ===")
print(f"Tasks: {CONFIG['num_tasks']} assays")
print(f"Dropout: encoder={CONFIG['dropout']}, fusion={CONFIG['fusion_dropout']}, head={CONFIG['head_dropout']}")
print(f"Weight decay: {CONFIG['weight_decay']}")
print(f"Head: {CONFIG['head_type']} (hidden={CONFIG['head_hidden_dim']})")
print(f"LR: {CONFIG['learning_rate']}, batch: {CONFIG['batch_size']}")
print(f"Scheduler: {CONFIG['scheduler']} (patience={CONFIG['scheduler_patience']}, factor={CONFIG['scheduler_factor']})")
print(f"Pool: {CONFIG['pool']}" + (f" (steps={CONFIG['set2set_processing_steps']})" if CONFIG['pool'] == 'set2set' else ""))
print(f"Gradient clipping: {CONFIG['grad_clip']}")
print(f"Text proj init: {CONFIG['text_proj_init']} (gain={CONFIG['text_proj_init_gain']})")
print(f"Freeze text proj: {CONFIG['freeze_text_proj']}")

=== Seed: 42 (reproducibility enabled) ===
=== Tox21 Multi-Label Configuration ===
Tasks: 12 assays
Dropout: encoder=0.3, fusion=0.3, head=0.6
Weight decay: 0.1
Head: mlp (hidden=64)
LR: 0.0003, batch: 64
Scheduler: cosine (patience=5, factor=0.5)
Pool: set2set (steps=6)
Gradient clipping: None
Text proj init: xavier (gain=0.1)
Freeze text proj: True


In [3]:
# === Load Tox21 Dataset ===
import deepchem as dc
from deepchem.molnet.load_function.tox21_datasets import TOX21_URL, TOX21_TASKS
import os

SPLIT_TYPE = "random"  # Options: "scaffold" (harder, realistic) or "random" (easier)

# Download if needed
data_dir = dc.utils.data_utils.get_data_dir()
dataset_file = os.path.join(data_dir, "tox21.csv.gz")
if not os.path.exists(dataset_file):
    dc.utils.data_utils.download_url(url=TOX21_URL, dest_dir=data_dir)

# Load CSV directly
df = pd.read_csv(dataset_file)
tasks = TOX21_TASKS

# Extract SMILES and labels
all_smiles = df['smiles'].tolist()
all_y = df[tasks].values.astype(np.float32)  # Shape: (N, 12), NaN for missing

# Filter valid SMILES
from rdkit import Chem
mols = [Chem.MolFromSmiles(s) for s in all_smiles]
valid_idx = [i for i, m in enumerate(mols) if m is not None]
valid_smiles = [all_smiles[i] for i in valid_idx]
valid_y = all_y[valid_idx]

# Split
if SPLIT_TYPE == "scaffold":
    from deepchem.splits import ScaffoldSplitter
    splitter = ScaffoldSplitter()
else:
    from deepchem.splits import RandomSplitter
    splitter = RandomSplitter()

train_idx, val_idx, test_idx = splitter.split(
    dc.data.NumpyDataset(X=np.zeros((len(valid_smiles), 1)), y=valid_y, ids=valid_smiles),
    frac_train=0.8, frac_valid=0.1, frac_test=0.1,
    **({"seed": CONFIG["seed"]} if SPLIT_TYPE == "random" else {})
)

train_smiles = [valid_smiles[i] for i in train_idx]
train_y = valid_y[train_idx]
valid_smiles = [valid_smiles[i] for i in val_idx]
valid_y = valid_y[val_idx]
test_smiles = [valid_smiles[i] for i in test_idx]
test_y = valid_y[test_idx]

print(f"Tasks ({len(tasks)}): {tasks}")
print(f"Split: {SPLIT_TYPE} | Train: {len(train_smiles)} | Valid: {len(valid_smiles)} | Test: {len(test_smiles)}")
print(f"Train labels shape: {train_y.shape}")

No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
No normalization for NumAmideBonds. Feature removed!
No normalization for NumAtomStereoCenters. Feature removed!
No normalization for NumBridgeheadAtoms. Feature removed!
No normalization for NumHeterocycles. Feature removed!
No normalization for NumSpiroAtoms. Feature removed!
No normalization for NumUnspecifiedAtomStereoCenters. Feature removed!
No normalization for Phi. Feature removed!
Skipped loading some Tensorflow models, missing a dependency. No module named 'tensorflow'
Skipped loading modules with pytorch-geometric dependency, missing a dependency. No module named 'torch_geometric'
Skipped loading modules with transformers dependency. No module named 'transformers'
cannot import name 'HuggingFaceModel' from 'deepchem.models.torch_models' (c:\Users\robsc\Home\Dev\molfusion2\.venv\Lib\site-packages\deepchem\models\torch_models\__init__.py)
Skipped loading modules with pytorch-geometric depe

IndexError: list index out of range

In [ ]:
# === Load Text Embeddings (from cache) ===
from utils.embedding_cache import EfficientEmbeddingCache

COT_TEXT_DIR = workspace_root / "cache" / "cot_texts"
COT_EMB_DIR  = workspace_root / "cache" / "cot_embeddings"

TASK = "toxicity_fast"  # Cache prefix — matches {task}_text_embeddings_compact.npz

# Load compact npz cache (run the conversion cell below first if only .pkl exists)
npz_path = COT_EMB_DIR / f"{TASK}_text_embeddings_compact.npz"
cache = EfficientEmbeddingCache.load(npz_path)

all_smiles = train_smiles + valid_smiles + test_smiles
all_emb = cache.get_batch(all_smiles)               # (N, 3072) numpy float32
all_emb_t = torch.from_numpy(all_emb)                # → torch tensor

n_train = len(train_smiles)
n_valid = len(valid_smiles)
train_text_emb = all_emb_t[:n_train]
valid_text_emb = all_emb_t[n_train:n_train + n_valid]
test_text_emb  = all_emb_t[n_train + n_valid:]

print(f"✓ Loaded from {npz_path.name} ({len(cache)} molecules)")
print(f"  Embeddings: train={train_text_emb.shape}, valid={valid_text_emb.shape}, test={test_text_emb.shape}")

In [ ]:
# === Initialize SEG for Multi-Label ===
from models import SEGPredictor, SEGPredictorConfig

seg_config = SEGPredictorConfig(
    task="classification",
    num_tasks=CONFIG["num_tasks"],  # Multi-label: 12 outputs
    hidden_channels=CONFIG["hidden_channels"],
    K=CONFIG["K"],
    num_layers=CONFIG["num_layers"],
    dropout=CONFIG["dropout"],
    pool=CONFIG["pool"],
    set2set_processing_steps=CONFIG["set2set_processing_steps"],
    text_embedding_dim=3072,
    text_projection_dim=CONFIG["text_projection_dim"],
    text_proj_init=CONFIG["text_proj_init"],
    text_proj_init_gain=CONFIG["text_proj_init_gain"],
    freeze_text_proj=CONFIG["freeze_text_proj"],
    fusion=CONFIG["fusion"],
    fusion_dim=CONFIG["fusion_dim"],
    fusion_n_heads=CONFIG["fusion_n_heads"],
    fusion_dropout=CONFIG["fusion_dropout"],
    head_type=CONFIG["head_type"],
    head_hidden_dim=CONFIG["head_hidden_dim"],
    head_dropout=CONFIG["head_dropout"],
)

seg = SEGPredictor(config=seg_config)
print(f"SEG initialized: {CONFIG['num_tasks']} tasks, {CONFIG['fusion']} fusion")
print(f"Text proj: init={CONFIG['text_proj_init']}, frozen={CONFIG['freeze_text_proj']}")

In [ ]:
# === Train SEG ===
print(f"Training SEG on Tox21 ({CONFIG['num_tasks']} tasks, seed={CONFIG['seed']})\n")

history = seg.fit(
    smiles_list=train_smiles,
    labels=train_y,  # Shape: (N, 12) with NaN
    val_smiles=valid_smiles,
    val_labels=valid_y,
    text_embeddings=train_text_emb,
    val_text_embeddings=valid_text_emb,
    num_epochs=CONFIG["num_epochs"],
    batch_size=CONFIG["batch_size"],
    learning_rate=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
    patience=CONFIG["patience"],
    scheduler=CONFIG["scheduler"],
    scheduler_patience=CONFIG["scheduler_patience"],
    scheduler_factor=CONFIG["scheduler_factor"],
    min_lr=CONFIG["min_lr"],
    grad_clip=CONFIG["grad_clip"],
    seed=CONFIG["seed"] or 0,
    verbose=True,
)

# Count SEG parameters
n_params_seg = sum(p.numel() for p in seg._encoder.parameters())
n_params_seg += sum(p.numel() for p in seg._fusion.parameters())
n_params_seg += sum(p.numel() for p in seg._head.parameters())
if seg._text_proj: n_params_seg += sum(p.numel() for p in seg._text_proj.parameters())
print(f"\nSEG: {n_params_seg:,} parameters")

In [ ]:
# === Test Set Evaluation (Per-Task AUC-ROC) ===
from sklearn.metrics import roc_auc_score

# Get predictions
test_preds = seg.predict_batch(test_smiles, text_embeddings=test_text_emb)
print(f"Predictions shape: {test_preds.shape}")  # Should be (N_test, 12)

# Calculate per-task AUC-ROC (handling NaN labels)
aucs = []
print("\n=== Per-Task Test AUC-ROC ===")
for i, task in enumerate(tasks):
    valid_mask = ~np.isnan(test_y[:, i])
    y_true = test_y[valid_mask, i]
    y_pred = test_preds[valid_mask, i]
    
    if len(np.unique(y_true)) < 2 or len(y_true) < 10:
        print(f"{task:15}: N/A (insufficient data)")
        aucs.append(np.nan)
        continue
    
    auc = roc_auc_score(y_true, y_pred)
    aucs.append(auc)
    print(f"{task:15}: {auc:.4f} (n={len(y_true)})")

# Mean AUC (excluding NaN)
valid_aucs = [a for a in aucs if not np.isnan(a)]
mean_auc = np.mean(valid_aucs)
print(f"\n{'Mean AUC':15}: {mean_auc:.4f} ({len(valid_aucs)} tasks)")
print(f"\nConfig: dropout={CONFIG['dropout']}, wd={CONFIG['weight_decay']}, fusion={CONFIG['fusion']}, pool={CONFIG['pool']}")